Top 10 recommended movies based on popularity

Angela Irungu - 669 289

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('DSA_4060_Week2_Movie_Ratings.csv')

# 1. Display first 10 records
print("--- First 10 Records ---")
print(df.head(10))

# 2. Dimensions and data types
print("\n--- Dataset Dimensions ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n--- Data Types ---")
print(df.dtypes)

# 3. Missing values and duplicates
print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Records Count ---")
print(f"Exact duplicates: {df.duplicated().sum()}")

--- First 10 Records ---
  user_id movie_id     movie_title  rating             timestamp
0    U001     M001  Nairobi Nights       3  2026-01-01T08:00:00Z
1    U004     M001  Nairobi Nights       4  2026-01-06T09:00:00Z
2    U007     M001  Nairobi Nights       2  2026-01-11T10:00:00Z
3    U010     M001  Nairobi Nights       2  2026-01-16T11:00:00Z
4    U013     M001  Nairobi Nights       4  2026-01-21T12:00:00Z
5    U016     M001  Nairobi Nights       3  2026-01-26T13:00:00Z
6    U019     M001  Nairobi Nights       3  2026-01-31T14:00:00Z
7    U022     M001  Nairobi Nights       2  2026-02-05T15:00:00Z
8    U025     M001  Nairobi Nights       2  2026-02-10T16:00:00Z
9    U028     M001  Nairobi Nights       4  2026-02-15T17:00:00Z

--- Dataset Dimensions ---
Rows: 509, Columns: 5

--- Data Types ---
user_id        object
movie_id       object
movie_title    object
rating          int64
timestamp      object
dtype: object

--- Missing Values ---
user_id        2
movie_id       1
movie_ti

Cleaning the dataset

In [2]:
# 1. Remove exact duplicate records
df_clean = df.drop_duplicates()

# 2. Remove records with missing user_id or movie_id
df_clean = df_clean.dropna(subset=['user_id', 'movie_id'])

# 3. Confirm ratings fall within expected 1 to 5 scale
df_clean = df_clean[df_clean['rating'].between(1, 5)]

print(f"Cleaned Dataset Shape: {df_clean.shape}")

Cleaned Dataset Shape: (500, 5)


In [3]:
# Exploratory calculations
total_ratings = len(df_clean)
num_users = df_clean['user_id'].nunique()
num_movies = df_clean['movie_id'].nunique()

most_active_user = df_clean['user_id'].value_counts().idxmax()
most_active_user_count = df_clean['user_id'].value_counts().max()

most_rated_movie_id = df_clean['movie_id'].value_counts().idxmax()
most_rated_movie_title = df_clean[df_clean['movie_id'] == most_rated_movie_id]['movie_title'].iloc[0]
most_rated_movie_count = df_clean['movie_id'].value_counts().max()

avg_rating = df_clean['rating'].mean()

print(f"Total Number of Ratings: {total_ratings}")
print(f"Number of Unique Users: {num_users}")
print(f"Number of Unique Movies: {num_movies}")
print(f"Most Active User: {most_active_user} ({most_active_user_count} ratings)")
print(f"Most-Rated Movie: {most_rated_movie_title} (ID: {most_rated_movie_id}) with {most_rated_movie_count} ratings")
print(f"Average Rating: {avg_rating:.2f}")

Total Number of Ratings: 500
Number of Unique Users: 50
Number of Unique Movies: 24
Most Active User: U010 (17 ratings)
Most-Rated Movie: Nairobi Nights (ID: M001) with 37 ratings
Average Rating: 3.45


 User-Item Interaction Matrix & Sparsity Calculation

In [4]:
# Handle duplicate ratings by taking average rating per (user, movie) pair
df_matrix_prep = df_clean.groupby(['user_id', 'movie_id'], as_index=False)['rating'].mean()

# Pivot table: Rows = Users, Columns = Movies
user_item_matrix = df_matrix_prep.pivot(index='user_id', columns='movie_id', values='rating')

# Display small section of the matrix (First 5 users x First 5 movies)
print("--- User-Item Matrix Preview ---")
print(user_item_matrix.iloc[:5, :5])

# Calculate Sparsity
# Formula: Sparsity = (1 - (Total Ratings / (Total Users * Total Movies))) * 100
total_possible_interactions = num_users * num_movies
sparsity = (1 - (total_ratings / total_possible_interactions)) * 100

print(f"\nSparsity Calculation: (1 - ({total_ratings} / ({num_users} * {num_movies}))) * 100")
print(f"Interaction Matrix Sparsity: {sparsity:.2f}%")

--- User-Item Matrix Preview ---
movie_id  M001  M002  M003  M004  M005
user_id                               
U001       3.0   2.0   NaN   NaN   4.0
U002       3.0   NaN   NaN   3.5   NaN
U003       NaN   NaN   NaN   NaN   4.0
U004       4.0   NaN   NaN   NaN   3.0
U005       2.0   4.0   NaN   3.0   NaN

Sparsity Calculation: (1 - (500 / (50 * 24))) * 100
Interaction Matrix Sparsity: 58.33%


Sparsity Interpretation
A sparsity rate of 58.33% means that 58.33% of all possible user-movie rating combinations are empty/unobserved.

In real-world commercial recommender systems (e.g., Netflix, Spotify), sparsity often exceeds 99% because millions of users interact with only a tiny fraction of available media.

A 58.33% sparsity level indicates a relatively dense dataset for experimentation, meaning users have evaluated a reasonable proportion of the catalog.

Building a popularity based recommender

In [5]:
# Aggregate average rating and total count per movie
movie_stats = df_clean.groupby(['movie_id', 'movie_title']).agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

# Filter for movies with at least 10 ratings and rank by highest average rating
top_10_recommendations = movie_stats[movie_stats['rating_count'] >= 10].sort_values(
    by=['avg_rating', 'rating_count'],
    ascending=[False, False]
).head(10)

# Display Top 10 Recommendations
print("--- Top 10 Recommended Movies (>= 10 Ratings) ---")
print(top_10_recommendations[['movie_id', 'movie_title', 'avg_rating', 'rating_count']].to_string(index=False))

# Export to CSV for submission
top_10_recommendations.to_csv('top_10_recommended_movies.csv', index=False)

--- Top 10 Recommended Movies (>= 10 Ratings) ---
movie_id         movie_title  avg_rating  rating_count
    M021          Market Day    4.466667            15
    M013 Midnight in Mombasa    4.352941            17
    M014       Code Breakers    4.250000            16
    M019       Future Africa    4.133333            15
    M020          Home Again    4.133333            15
    M007         Hidden Lake    4.125000            16
    M006    City of Tomorrow    3.911765            34
    M005     Mountain Echoes    3.864865            37
    M012      The Final Goal    3.764706            17
    M004      Coastal Dreams    3.666667            27


Interpretation & Findings

Why These Movies Appear at the Top:These top 10 movies achieved the highest mean ratings among candidate movies meeting the minimum threshold of $\ge 10$ interactions. Filtering out titles with low review volume prevents niche titles with 1 or 2 lucky 5-star ratings from skewing the rankings.

Key Advantage:

Solves Cold-Start for New Users: Popularity recommender systems do not require user demographic data or historical preference logs. They serve as an immediate default recommendation for guest users or brand-new accounts.


Two Key Limitations:

Lack of Personalization: Every user receives identical recommendations regardless of individual genre preferences, previous viewing history, or demographics.

Popularity Bias / Over-exposure: Popular titles continuously collect more views and reviews, creating a feedback loop that suppresses hidden gems in the long tail of the catalog.